#### IMPORTS

In [1]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
import os

/u/gongt/conda-envs/spikeinterface/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rec_dir = '/ptmp/gongt/GLC6962_l_g1/preprocessed'
ks4_dir = '/ptmp/gongt/GLC6962_l_g1/KS4/test_MG_params2/sorter_output'
rec = si.load(rec_dir)
sorting = si.read_kilosort(ks4_dir)  # reads from sorter_output/ automatically
print(f"Sampling rate: {rec.get_sampling_frequency()} Hz")
print(f"Units: {sorting.get_unit_ids()[:10]} ...")

Sampling rate: 30000.0 Hz
Units: [ 0  1  2  4  5  6  7 10 13 14] ...


In [3]:
channels = rec.get_num_channels()
fs = rec.get_sampling_frequency()
print(channels)
print(fs)

352
30000.0


In [4]:
print(sorting)

KiloSortSortingExtractor: 340 units - 1 segments - 30.0kHz


In [14]:
unit_mapping = {
    0: 13, 1: 14, 2: 268, 3: 280, 4: 306, 5: 387, 6: 494, 7: 583, 
    8: 650, 9: 710, 10: 714, 11: 721, 12: 740, 13: 742, 14: 753, 
    15: 754, 16: 759, 17: 775, 18: 807, 19: 811, 20: 820, 21: 831, 
    22: 833, 23: 843, 24: 851, 25: 863, 26: 874, 27: 878, 28: 885, 
    29: 892, 30: 897, 31: 903, 32: 908, 33: 914, 34: 919, 35: 936, 
    36: 941, 37: 946, 38: 950
}
unit_indices = sorted(unit_mapping.keys())


In [8]:
# Cell 2: Parameters — set these
unit_indice = 27
unit_id   = unit_mapping[unit_indice]       # the unit you care about
timepoint_s = 1568.47  # timepoint in SECONDS; or use timepoint_samples below
n_ch      = 32       # number of nearest channels
win_ms    = 3.0      # waveform window: ±win_ms/2 around spike

# Find the closest spike to your timepoint
spike_times = sorting.get_unit_spike_train(unit_id, segment_index=0)  # in samples
print(f"Unit {unit_id} has {len(spike_times)} spikes.")
timepoint_samples = int(timepoint_s * fs)

# nearest spike to the requested timepoint
idx_nearest = np.argmin(np.abs(spike_times - timepoint_samples))
center_sample = spike_times[idx_nearest]

print(f"Requested timepoint: {timepoint_s:.4f} s  →  sample {timepoint_samples}")
print(f"Nearest spike:       {center_sample / fs:.4f} s  →  sample {center_sample}")
print(f"Δt = {(center_sample - timepoint_samples)/fs*1000:.2f} ms")


Unit 878 has 415192 spikes.
Requested timepoint: 1568.4700 s  →  sample 47054100
Nearest spike:       1568.4569 s  →  sample 47053708
Δt = -13.07 ms


In [22]:
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import warnings

# ──────────────────────────────────────────────────────────────
#  SpikeInterface Import
# ──────────────────────────────────────────────────────────────
try:
    import spikeinterface.full as si
except ImportError:
    raise ImportError("This script requires spikeinterface. Please install it via 'pip install spikeinterface[full]'")

# ══════════════════════════════════════════════════════════════
#  CONFIG – edit this block to fine-tune spacing
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # ── paths ─────────────────────────────────────────────────
    recording_dir   = rec_dir,
    ks_dir          = ks4_dir, 
 
    # ── window ────────────────────────────────────────────────
    start_time_s    = 1568.47,   # seconds from recording start
    window_s        = 8.0,       # seconds to display
 
    # ── units to inspect ──────────────────────────────────────
    unit_indices    = [15,16,33,14,10,32,4,13,12,31,11,0,30,1,2,3],   
 
    # ── waveform panel spacing & layout ───────────────────────
    n_nearest_ch         = 10,     # channels around best channel
    samples_before       = 30,     # ~1 ms before spike sample 
    samples_after        = 50,     # ~1.6 ms after spike sample 
    waveform_gain        = 0.5,    # Lower slightly if peaks overlap vertically
    waveforms_fig_height = 50.0,   # INCREASE THIS to make the gap between channels bigger
 
    # ── trace panel ───────────────────────────────────────────
    filter_trace    = False,   
    filter_lo       = 300,    
    filter_hi       = 6000,   
    trace_color     = "0.4",  
 
    # ── output ────────────────────────────────────────────────
    fig1_out        = "waveforms.png",
    fig2_out        = "traces.png",
    dpi             = 150,
)
# ══════════════════════════════════════════════════════════════
 
# ──────────────────────────────────────────────────────────────
#  Helpers
# ──────────────────────────────────────────────────────────────
 
def load_npy(path, name):
    f = Path(path) / name
    if not f.exists():
        return None
    return np.load(f)
 
def nearest_channels(ch_positions, best_ch_idx, n=32):
    dists = np.linalg.norm(ch_positions - ch_positions[best_ch_idx], axis=1)
    order = np.argsort(dists)
    return order[:n]
 
def get_cluster_best_channel(cluster_id, spike_clusters, spike_templates,
                              templates, channel_map):
    mask = spike_clusters == cluster_id
    if mask.sum() == 0:
        return 0
    tmpl_indices = spike_templates[mask]
    unique_t, counts_t = np.unique(tmpl_indices, return_counts=True)
    dominant_tmpl = unique_t[np.argmax(counts_t)]
    wf = templates[dominant_tmpl]          
    amplitudes = wf.max(axis=0) - wf.min(axis=0)
    best_local = int(np.argmax(amplitudes))
    return best_local
 
def make_color_palette(n):
    cmap = cm.get_cmap("tab20" if n > 10 else "tab10")
    return [cmap(i / max(n - 1, 1)) for i in range(n)]
 
# ──────────────────────────────────────────────────────────────
#  Main
# ──────────────────────────────────────────────────────────────
 
def main(cfg):
    ks_dir = Path(cfg["ks_dir"])
    rec_dir = Path(cfg["recording_dir"])
 
    print(f"Loading recording from {rec_dir} using SpikeInterface...")
    recording = si.load_extractor(rec_dir)
    sr = recording.get_sampling_frequency()
    si_ch_ids = list(recording.get_channel_ids())
    si_positions = recording.get_channel_locations()
    
    start_s   = cfg["start_time_s"]
    win_s     = cfg["window_s"]
    start_smp = int(start_s * sr)
    win_smp   = int(win_s * sr)
    end_smp   = start_smp + win_smp
 
    print(f"Loading Kilosort outputs from {ks_dir}...")
    spike_times     = load_npy(ks_dir, "spike_times.npy")
    spike_clusters  = load_npy(ks_dir, "spike_clusters.npy")
    spike_templates_arr = load_npy(ks_dir, "spike_templates.npy")
    templates       = load_npy(ks_dir, "templates.npy")        
    channel_map     = load_npy(ks_dir, "channel_map.npy")
    ch_pos          = load_npy(ks_dir, "channel_positions.npy")  
 
    pre        = cfg["samples_before"]
    post       = cfg["samples_after"]
    wf_len     = pre + post
 
    unit_indices   = cfg["unit_indices"]
    master_colors  = make_color_palette(len(unit_indices))
    idx_to_color   = {u_idx: master_colors[i] for i, u_idx in enumerate(unit_indices)}
 
    ks_to_si_idx = []
    for i in range(len(ch_pos)):
        dists = np.linalg.norm(si_positions - ch_pos[i], axis=1)
        ks_to_si_idx.append(np.argmin(dists))
    ks_to_si_idx = np.array(ks_to_si_idx)
 
    print(f"Extracting traces...")
    neural_raw = recording.get_traces(start_frame=start_smp, end_frame=end_smp, return_scaled=True)
    neural_raw = neural_raw.T  
    nChan = neural_raw.shape[0]
 
    if cfg["filter_trace"]:
        print("Applying bandpass filtering...")
        from scipy.signal import butter, filtfilt
        nyq = sr / 2
        b, a = butter(3, [cfg["filter_lo"] / nyq, cfg["filter_hi"] / nyq], btype="band")
        neural_filt = filtfilt(b, a, neural_raw, axis=1).astype(np.float32)
    else:
        neural_filt = neural_raw
 
    in_window = (spike_times > start_smp) & (spike_times < end_smp)
    win_times    = spike_times[in_window]
    win_clusters = spike_clusters[in_window]
 
    # ──────────────────────────────────────────────────────────
    #  PRE-FILTER: Drop empty units
    # ──────────────────────────────────────────────────────────
    active_unit_indices = []
    for u_idx in unit_indices:
        ks_id = unit_mapping[u_idx]
        if np.sum(win_clusters == ks_id) > 0:
            active_unit_indices.append(u_idx)
            
    n_active = len(active_unit_indices)
    if n_active == 0:
        print("Warning: No spikes found for any specified units in this window! Skipping Figure 1.")
    else:
        # ═════════════════════════════════════════════════════════
        #  FIGURE 1 – High-Stretched Waveform Grid
        # ═════════════════════════════════════════════════════════
        fig1, axes1 = plt.subplots(1, n_active, figsize=(2.5 * n_active, cfg["waveforms_fig_height"]), sharey=True, squeeze=False)
        fig1.suptitle(f"Waveforms Spaced by Probe Depth (Phy Method)", color="w", y=0.99, fontsize=12)
        fig1.patch.set_facecolor("k")
     
        all_active_y_positions = []
        unit_channel_groups = {}
        
        for u_idx in active_unit_indices:
            ks_id = unit_mapping[u_idx]
            best_local = get_cluster_best_channel(ks_id, spike_clusters, spike_templates_arr, templates, channel_map)
            nn_idx = nearest_channels(ch_pos, best_local, n=cfg["n_nearest_ch"])
            unit_channel_groups[u_idx] = nn_idx
            all_active_y_positions.extend(ch_pos[nn_idx, 1])
            
        min_active_y = min(all_active_y_positions) - 30
        max_active_y = max(all_active_y_positions) + 30
     
        for i, u_idx in enumerate(active_unit_indices):
            ks_id = unit_mapping[u_idx]
            ax = axes1[0, i]
            ax.set_facecolor("k")
            color_u = idx_to_color[u_idx]
     
            mask_u = win_clusters == ks_id
            spike_smp_u = win_times[mask_u]   
            nn_idx = unit_channel_groups[u_idx]
            t_axis = (np.arange(wf_len) - pre) / sr * 1000   # ms
            
            # Draw sparse, subtle baseline references
            for ch in nn_idx:
                y_pos = ch_pos[ch, 1]
                ax.axhline(y_pos, color="0.25", lw=0.4, ls="-", alpha=0.3)
                ax.text(t_axis[0], y_pos + 1.5, f"Ch{ch}", color="0.5", fontsize=6.5, va="bottom")
     
            alpha = max(0.12, min(0.6, 1.2 / (len(spike_smp_u) ** 0.3)))
            for sp in spike_smp_u:
                offset = int(sp) - start_smp
                if offset - pre < 0 or offset + post > win_smp:
                    continue
                
                si_indices = ks_to_si_idx[nn_idx]
                snip = neural_filt[si_indices, :][:, offset - pre : offset + post]
                
                for ch_snip, ch_global_idx in zip(snip, nn_idx):
                    y_pos = ch_pos[ch_global_idx, 1]
                    ax.plot(t_axis, y_pos + (ch_snip * cfg["waveform_gain"]), color=color_u, alpha=alpha, lw=0.5)
     
            ax.set_title(f"Idx {u_idx}\n[KS {ks_id}]\n({len(spike_smp_u)} spks)", color=color_u, fontsize=9)
            ax.tick_params(colors="0.6", labelsize=8)
            ax.set_xlabel("ms", color="0.5", fontsize=7.5)
            ax.grid(False)
            for sp_side in ax.spines.values():
                sp_side.set_edgecolor("0.15")
     
        axes1[0, 0].set_ylabel("Probe Depth (µm)", color="w", fontsize=10)
        axes1[0, 0].set_ylim(min_active_y, max_active_y)
     
        plt.tight_layout()
        fig1.savefig(cfg["fig1_out"], dpi=cfg["dpi"], bbox_inches="tight", facecolor="k")
        print(f"Saved High-Separation Depth Waveforms: {cfg['fig1_out']}")
 
    # ═════════════════════════════════════════════════════════
    #  FIGURE 2 – continuous trace ordered and labeled by depth
    # ═════════════════════════════════════════════════════════
    depth_order_idx = np.argsort(ch_pos[:, 1])

    fig2, ax2 = plt.subplots(figsize=(20, max(8, nChan * 0.18)))
    ax2.set_facecolor("k")
    fig2.patch.set_facecolor("k")

    t_win = np.linspace(start_s, start_s + win_s, win_smp)

    ch_range = np.percentile(np.abs(neural_filt), 99, axis=1)
    trace_spacing = float(np.median(ch_range)) * 6.0
    if trace_spacing < 1:
        trace_spacing = 50.0

    print("Plotting continuous traces ordered and stacked by depth...")
    for ci in range(nChan):
        orig_ks_idx = depth_order_idx[ci]
        orig_si_idx = ks_to_si_idx[orig_ks_idx]
        offset_ci = ci * trace_spacing
        ax2.plot(t_win, neural_filt[orig_si_idx, :] + offset_ci, color=cfg["trace_color"], lw=0.3, alpha=0.5, rasterized=True)

    print("Overlaying verified spike dots at true channel locations …")
    for u_idx in unit_indices:
        ks_id = unit_mapping[u_idx]
        color_u = idx_to_color[u_idx]
        
        mask_u = win_clusters == ks_id
        spike_smp_u = win_times[mask_u]
        if len(spike_smp_u) == 0:
            continue

        if templates is not None and spike_templates_arr is not None:
            best_local_ks = get_cluster_best_channel(ks_id, spike_clusters, spike_templates_arr, templates, channel_map)
        else:
            best_local_ks = 0  

        depth_ordered_idx = np.where(depth_order_idx == best_local_ks)[0][0]
        y_dot_base = depth_ordered_idx * trace_spacing
        orig_si_idx = ks_to_si_idx[best_local_ks]
        
        spike_offsets = (spike_smp_u - start_smp).astype(int)
        valid_masks = (spike_offsets >= 0) & (spike_offsets < win_smp)
        spike_offsets = spike_offsets[valid_masks]
        
        if len(spike_offsets) > 0:
            t_sp = start_s + spike_offsets / sr
            y_val = neural_filt[orig_si_idx, spike_offsets] + y_dot_base
            
            ax2.scatter(t_sp, y_val, color=color_u, s=24, zorder=5, edgecolors='w', linewidths=0.3)
            first_idx = np.argmin(t_sp)
            ax2.text(t_sp[first_idx] + 0.004, y_val[first_idx], f"Idx {u_idx}", 
                     color=color_u, fontsize=8, weight='bold', va='center',
                     bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1), zorder=6)

    legend_handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=idx_to_color[u_idx], 
                                markersize=6, label=f"Idx {u_idx} (KS {unit_mapping[u_idx]})", lw=0)
                      for u_idx in unit_indices]
    ax2.legend(handles=legend_handles, loc="upper right", framealpha=0.5, fontsize=8, labelcolor="w", facecolor="0.1")

    tick_step = max(1, nChan // 30)
    yticks = [ci * trace_spacing for ci in range(0, nChan, tick_step)]
    ytick_labels = [f"{ch_pos[depth_order_idx[ci], 1]} µm (Ch {depth_order_idx[ci]})" for ci in range(0, nChan, tick_step)]
    
    ax2.set_yticks(yticks)
    ax2.set_yticklabels(ytick_labels, fontsize=7, color="w")
    ax2.set_xlabel("Time (s)", color="w", fontsize=9)
    ax2.set_ylabel("Physical Depth along Probe Channel Map", color="w", fontsize=9)
    ax2.set_xlim(start_s, start_s + win_s)
    ax2.set_title(f"Continuous Traces (Labeled by True Depth) | t = {start_s:.2f}–{start_s+win_s:.2f} s", color="w", fontsize=10)
    ax2.tick_params(colors="w")
    for sp_side in ax2.spines.values():
        sp_side.set_edgecolor("0.3")

    plt.tight_layout()
    fig2.savefig(cfg["fig2_out"], dpi=cfg["dpi"], bbox_inches="tight", facecolor="k")
    print(f"Saved Realigned Traces: {cfg['fig2_out']}")
    plt.show()
    print("Done.")
 
if __name__ == "__main__":
    main(CFG)

Loading recording from /ptmp/gongt/GLC6962_l_g1/preprocessed using SpikeInterface...
Loading Kilosort outputs from /ptmp/gongt/GLC6962_l_g1/KS4/test_MG_params2/sorter_output...


/tmp/ipykernel_15899/1028923238.py:93: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  recording = si.load_extractor(rec_dir)
/tmp/ipykernel_15899/1028923238.py:81: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("tab20" if n > 10 else "tab10")
/tmp/ipykernel_15899/1028923238.py:127: DeprecationWarning: `return_scaled` is deprecated and will be removed in version 0.105.0. Use `return_in_uV` instead.
  neural_raw = recording.get_traces(start_frame=start_smp, end_frame=end_smp, return_scaled=True)


Extracting traces...


/tmp/ipykernel_15899/1028923238.py:160: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig1, axes1 = plt.subplots(1, n_active, figsize=(2.5 * n_active, cfg["waveforms_fig_height"]), sharey=True, squeeze=False)


Saved High-Separation Depth Waveforms: waveforms.png
Plotting continuous traces ordered and stacked by depth...
Overlaying verified spike dots at true channel locations …
Saved Realigned Traces: traces.png
Done.


/tmp/ipykernel_15899/1028923238.py:299: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
